In [ ]:
import pandas as pd
import numpy as np
import xarray as xr
import geopandas as gpd
import dask.dataframe as dd
import dask
import coiled
import os
import glob
from global_snowmelt_runoff_onset.config import Config

In [ ]:
config = Config('config/global_config_v9.txt')

In [ ]:
# cluster = coiled.Cluster(
#     name="parquets_to_aggregation_netcdfs",
#     idle_timeout="10 minutes",
#     n_workers=50,
#     worker_memory="32 GB",
#     worker_cpu=4,
#     scheduler_memory="128 GB",
#     spot_policy="spot",
#     environ={"GDAL_DISABLE_READDIR_ON_OPEN": "EMPTY_DIR"},
#     workspace="uwtacolab",
# )

cluster = coiled.Cluster(
    name="parquets_to_aggregation_netcdfs",
    idle_timeout="10 minutes",
    n_workers=50,
    worker_memory="16 GB",
    #worker_cpu=4,
    #scheduler_memory="128 GB",
    spot_policy="spot",
    environ={"GDAL_DISABLE_READDIR_ON_OPEN": "EMPTY_DIR"},
    workspace="uwtacolab",
)
client = cluster.get_client()

In [ ]:
client.restart()

In [ ]:
def process_and_save_geographic_unit(unit_id, unit_id_column, parquet_path, filesystem, 
                                   dem_bin_low=0, dem_bin_high=9000, 
                                   dem_bin_interval=100, aspect_bin_interval=15):
    """
    Generic function to process single geographic unit and return netCDF dataset
    
    Parameters:
    -----------
    unit_id : int
        ID of the geographic unit to process
    unit_id_column : str
        Column name containing the unit IDs (e.g., 'GMBA_V2_ID', 'PFAF_ID')
    parquet_path : str
        Path to the parquet dataset
    filesystem : filesystem object
        Filesystem to use for reading data
    """
    
    # Setup bins and coordinates
    dem_bins = np.arange(dem_bin_low, dem_bin_high+dem_bin_interval, dem_bin_interval)
    aspect_bins = np.arange(0, 360+aspect_bin_interval, aspect_bin_interval)
    water_years = range(2015, 2025)
    dem_coords = dem_bins[:-1] + dem_bin_interval/2
    aspect_coords = aspect_bins[:-1] + aspect_bin_interval/2
    stat_coords = ["mean", "median", "count"]

    # Read data
    cols_needed = [unit_id_column, 'dem', 'aspect', 'runoff_onset_median', 'runoff_onset_mad'] + \
                  [f'runoff_onset_WY{year}' for year in water_years]
    
    unit_ddf = dd.read_parquet(
        parquet_path,
        filesystem=filesystem,
        columns=cols_needed,
        filters=[(unit_id_column, '==', unit_id)]
    )

    # Create bins and calculate anomalies
    def bin_and_anomaly(df):
        df = df.dropna(subset=['dem', 'aspect'])
        df['dem_bin'] = pd.cut(df['dem'], dem_bins).apply(
            lambda x: x.left+(dem_bin_interval/2) if pd.notnull(x) else np.nan
        )
        df['aspect_bin'] = pd.cut(df['aspect'], aspect_bins).apply(
            lambda x: x.left+(aspect_bin_interval/2) if pd.notnull(x) else np.nan
        )
        df['dem_bin'] = df['dem_bin'].astype('Int64')
        df['aspect_bin'] = df['aspect_bin'].astype('Float64')
        
        # Calculate anomalies
        for year in water_years:
            col = f'runoff_onset_WY{year}'
            anom_col = f'runoff_onset_anomaly_WY{year}'
            df[anom_col] = df[col].where(df[col] > 0) - df['runoff_onset_median']
        
        return df

    unit_ddf = unit_ddf.map_partitions(bin_and_anomaly)
    unit_ddf = unit_ddf.dropna(subset=['dem_bin', 'aspect_bin'])
    unit_ddf = unit_ddf.compute()

    # Calculate static statistics
    agg_static = unit_ddf.groupby(['dem_bin', 'aspect_bin']).agg({
        'runoff_onset_median': ['mean', 'median', 'count'],
        'runoff_onset_mad': ['mean', 'median', 'count']
    })#.compute()

    # Calculate yearly aggregations
    yearly_aggs = []
    anomaly_aggs = []
    
    for year in water_years:
        year_col = f'runoff_onset_WY{year}'
        # Filter out invalid values
        year_data = unit_ddf[unit_ddf[year_col] > 0][['dem_bin', 'aspect_bin', year_col]]
        
        year_agg = year_data.groupby(['dem_bin', 'aspect_bin']).agg({
            year_col: ['mean', 'median', 'count']
        })#.compute()
        
        year_agg.columns = ['mean', 'median', 'count']
        year_agg = year_agg.reset_index()
        year_agg['water_year'] = year
        yearly_aggs.append(year_agg)

        # Process anomaly data
        anom_col = f'runoff_onset_anomaly_WY{year}'
        anom_data = unit_ddf.dropna(subset=[anom_col])[['dem_bin', 'aspect_bin', anom_col]]
        
        anom_agg = anom_data.groupby(['dem_bin', 'aspect_bin']).agg({
            anom_col: ['mean', 'median', 'count']
        })#.compute()
        
        anom_agg.columns = ['mean', 'median', 'count']
        anom_agg = anom_agg.reset_index()
        anom_agg['water_year'] = year
        anomaly_aggs.append(anom_agg)

    # Convert to pandas DataFrames
    yearly_aggs = pd.concat(yearly_aggs, ignore_index=True)
    anomaly_aggs = pd.concat(anomaly_aggs, ignore_index=True)
    
    # Create dataset
    ds = xr.Dataset(
        coords={
            'elevation': dem_coords,
            'aspect': aspect_coords,
            'statistic': stat_coords,
            'water_year': list(water_years)
        }
    )

    # Initialize arrays
    shape_static = (len(ds.elevation), len(ds.aspect), len(ds.statistic))
    shape_yearly = shape_static + (len(ds.water_year),)

    ds['runoff_onset_median'] = xr.DataArray(np.full(shape_static, np.nan), 
        dims=('elevation', 'aspect', 'statistic'))
    ds['runoff_onset_mad'] = xr.DataArray(np.full(shape_static, np.nan), 
        dims=('elevation', 'aspect', 'statistic'))
    ds['runoff_onset'] = xr.DataArray(np.full(shape_yearly, np.nan), 
        dims=('elevation', 'aspect', 'statistic', 'water_year'))
    ds['runoff_onset_anomaly'] = xr.DataArray(np.full(shape_yearly, np.nan), 
        dims=('elevation', 'aspect', 'statistic', 'water_year'))

    # Fill values using pandas indexing
    for dem in ds.elevation.values:
        for asp in ds.aspect.values:
            # Fill static variables
            static_mask = (agg_static.index.get_level_values('dem_bin') == dem) & \
                         (agg_static.index.get_level_values('aspect_bin') == asp)
            
            if static_mask.any():
                static_data = agg_static[static_mask]
                for stat in ['mean', 'median', 'count']:
                    ds['runoff_onset_median'].loc[{
                        'elevation': dem,
                        'aspect': asp,
                        'statistic': stat
                    }] = static_data[('runoff_onset_median', stat)].iloc[0]
                    
                    ds['runoff_onset_mad'].loc[{
                        'elevation': dem,
                        'aspect': asp,
                        'statistic': stat
                    }] = static_data[('runoff_onset_mad', stat)].iloc[0]
            
            # Fill yearly variables
            year_mask = (yearly_aggs['dem_bin'] == dem) & (yearly_aggs['aspect_bin'] == asp)
            anom_mask = (anomaly_aggs['dem_bin'] == dem) & (anomaly_aggs['aspect_bin'] == asp)
            
            for stat in ['mean', 'median', 'count']:
                year_data = yearly_aggs[year_mask]
                if not year_data.empty:
                    for _, row in year_data.iterrows():
                        ds['runoff_onset'].loc[{
                            'elevation': dem,
                            'aspect': asp,
                            'statistic': stat,
                            'water_year': row['water_year']
                        }] = row[stat]
                
                anom_data = anomaly_aggs[anom_mask]
                if not anom_data.empty:
                    for _, row in anom_data.iterrows():
                        ds['runoff_onset_anomaly'].loc[{
                            'elevation': dem,
                            'aspect': asp,
                            'statistic': stat,
                            'water_year': row['water_year']
                        }] = row[stat]

    # Add attributes
    ds.elevation.attrs['units'] = 'meters'
    ds.aspect.attrs['units'] = 'degrees'
    ds.runoff_onset.attrs['units'] = 'day of water year'
    ds.runoff_onset_anomaly.attrs['units'] = 'days'
    ds.runoff_onset_median.attrs['units'] = 'day of water year'
    ds.runoff_onset_mad.attrs['units'] = 'days'
    
    ds.attrs['location'] = unit_id
    ds.attrs['unit_type'] = unit_id_column

    return ds.compute()

In [ ]:
@dask.delayed
def get_unique_ids(parquet_path, filesystem, id_column):
    """Get unique IDs from parquet dataset"""
    unique_ids = dd.read_parquet(
        parquet_path, 
        filesystem=filesystem,
        columns=[id_column],
    )[id_column].unique().compute()
    return unique_ids
    
def process_all_units(parquet_path, output_dir, filesystem, unit_type, 
                     id_column, batch_size=30):
    """
    Process all geographic units in batches
    
    Parameters:
    -----------
    parquet_path : str
        Path to parquet dataset
    output_dir : str
        Directory to save output files
    filesystem : filesystem object
        Filesystem for reading data
    unit_type : str
        Type of unit ('mountain_range' or 'river_basin')
    id_column : str
        Column name for unit IDs
    batch_size : int
        Number of units to process per batch
    """
    os.makedirs(output_dir, exist_ok=True)

    # Get all unit IDs
    unit_ids = get_unique_ids(parquet_path, filesystem, id_column).compute()

    # remove unit id of -9999
    unit_ids = unit_ids[unit_ids != -9999]

    # Filter out already processed units
    unprocessed_units = [
        unit_id for unit_id in unit_ids 
        if not os.path.exists(f"{output_dir}/{unit_type}_{unit_id}.nc")
    ]

    print(f"Found {len(unprocessed_units)} unprocessed {unit_type}s")

    # Process in batches
    for i in range(0, len(unprocessed_units), batch_size):
        batch = unprocessed_units[i:i + batch_size]
        futures = []

        print(f"Processing batch {i//batch_size + 1} of {(len(unprocessed_units)-1)//batch_size + 1}")
        
        # Submit batch of tasks
        for unit_id in batch:
            future = client.submit(
                process_and_save_geographic_unit, 
                unit_id, id_column, parquet_path, filesystem
            )
            futures.append(future)

        # Wait for batch completion and save results
        for future, result in dask.distributed.as_completed(futures, with_results=True):
            unit_id = result.attrs['location']
            output_file = f"{output_dir}/{unit_type}_{unit_id}.nc"
            result.to_netcdf(output_file)
            print(f"Saved to {output_file}")

        # Restart client between batches to clear memory
        client.restart()

In [ ]:
DATASETS = {
    'fcf_lte_50': f'snowmelt/analysis/parquets/full_datasets/fcf_lte_50/{config.version}',
    #'full_dataset': f'snowmelt/analysis/parquets/full_datasets/full_dataset/{config.version}', 
    #'no_trees': f'snowmelt/analysis/parquets/full_datasets/no_trees/{config.version}'
}

UNIT_CONFIGS = {
    'mountain_ranges': {
        'id_column': 'GMBA_V2_ID',
        'output_prefix': 'mountain_range'
    },
    'river_basins': {
        'id_column': 'PFAF_ID', 
        'output_prefix': 'river_basin'
    }
}

In [ ]:
client.restart()

In [ ]:
# print("Processing Mountain Ranges...")
# for dataset_name, base_path in DATASETS.items():
#     print(f"\n=== Processing {dataset_name} ===")
    
#     parquet_path = base_path
#     output_dir = f"aggregated_results/mountain_ranges/{dataset_name}/{config.version}"
    
#     client.restart()
#     process_all_units(
#         parquet_path=parquet_path,
#         output_dir=output_dir, 
#         filesystem=config.azure_blob_fs,
#         unit_type='mountain_range',
#         id_column='GMBA_V2_ID',
#         batch_size=20
#     )

In [ ]:
print("Processing River Basins...")
for dataset_name, base_path in DATASETS.items():
    print(f"\n=== Processing {dataset_name} ===")

    parquet_path = base_path  
    output_dir = f"aggregated_results/river_basins/{dataset_name}/{config.version}"

    client.restart()
    process_all_units(
        parquet_path=parquet_path,
        output_dir=output_dir,
        filesystem=config.azure_blob_fs, 
        unit_type='river_basin',
        id_column='PFAF_ID',
        batch_size=30
    )